# Práctica 4: Similitud Léxica y Semántica

## Introducción Teórica y Objetivos

El objetivo de la práctica es entrenar y evaluar modelos de **embeddings distribucionales (estáticos)** y compararlos con **embeddings contextuales** para tareas de similitud semántica en español.

Los **Word Embeddings** permiten representar palabras como vectores numéricos basándose en su contexto. Siguiendo la **Hipótesis Distribucional** de Firth (1957): *"Conocerás una palabra por las compañías que mantiene"*, las palabras que aparecen en contextos similares tenderán a tener significados similares.

### Tipos de embeddings

**Estáticos (Word2Vec, fastText):** Cada palabra tiene siempre el mismo vector independientemente del contexto. Word2Vec (Mikolov, Google 2013) propone dos arquitecturas:
- **CBOW**: predice la palabra central a partir de las palabras de contexto.
- **Skip-gram**: predice las palabras de contexto a partir de la palabra central.



**fastText** (Facebook, 2016) mejora Word2Vec representando cada palabra como la suma de los embeddings de sus n-gramas de caracteres, permitiendo gestionar palabras fuera del vocabulario (OOV).

**Contextuales (BERT):** Generan representaciones dinámicas basadas en la oración completa, resolviendo problemas como la polisemia.

### Evaluación
- **Intrínseca (Multi-SimLex)**: Calculamos la similitud coseno entre los vectores y la comparamos con las puntuaciones humanas mediante la **correlación de Spearman**.
- **Extrínseca (Spanish STS)**: Calculamos la similitud semántica de pares de frases y evaluamos con la **correlación de Pearson**.

### Datasets

| Dataset | Descripción | Uso | Métrica |
| :--- | :--- | :--- | :--- |
| **Multi-SimLex (ES)** | Pares de palabras con puntuaciones de similitud | Evaluación intrínseca | Spearman |
| **Spanish STS** | Pares de frases con puntuaciones de similitud | Evaluación extrínseca | Pearson |
| **raw.es (Wikicorpus)** | Corpus de Wikipedia en español | Entrenamiento de embeddings | — |

## 0. Setup

In [12]:
# Instalar dependencias si hace falta
#%pip install pandas scipy datasets tqdm gensim torch sentence-transformers scikit-learn transformers matplotlib

import pandas as pd
import numpy as np
import os
import re
from itertools import product
from scipy.stats import spearmanr, pearsonr
from datasets import load_dataset
from gensim.models import Word2Vec, FastText as GensimFastText
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

Device: cpu


## 1. Carga de Datasets

Esta práctica utiliza tres datasets:

| Dataset | Descripción | Uso |
|---|---|---|
| **Multi-SimLex** | Pares de palabras con puntuaciones de similitud semántica | Evaluación intrínseca |
| **Spanish STS** | Pares de frases con puntuaciones de similitud semántica | Evaluación extrínseca |
| **raw_es** | Corpus de Wikipedia en español | Entrenamiento de embeddings |

A partir de `raw_es`, entrenaremos y compararemos modelos de **Word2Vec** y **fastText**.

## Columnas del dataset Multi-SimLex (ES)

| Columna | Descripción |
|---|---|
| **PAIR_ID** | Identificador único del par de palabras en el dataset. |
| **SPA_W1** | Primera palabra del par. |
| **SPA_W2** | Segunda palabra del par. |
| **SPA_AVG** | Puntuación media de similitud semántica asignada por varios anotadores humanos. Cuanto mayor es el valor, más similares se consideran las dos palabras. En este dataset los valores observados van aproximadamente de **0.0** a **5.9**. |

In [13]:
# ============================================================
# DATASET 1 — Multi-SimLex  (reemplaza la celda original)
# ============================================================

os.system("wget -O SPA.csv https://web.archive.org/web/20231020014354/https://multisimlex.com/data/SPA.csv")

spa = pd.read_csv("SPA.csv")

annotator_cols = [c for c in spa.columns if c.startswith('Annotator')]
spa['SPA_AVG'] = spa[annotator_cols].mean(axis=1)

# Guardamos también las columnas de anotadores para el análisis posterior
multi_simlex = spa[['ID', 'Word 1', 'Word 2'] + annotator_cols + ['SPA_AVG']].copy()
multi_simlex.columns = ['PAIR_ID', 'SPA_W1', 'SPA_W2'] + annotator_cols + ['SPA_AVG']

print(f"Multi-SimLex (ES) — {len(multi_simlex)} pares de palabras")
print(f"Columnas de anotadores: {annotator_cols}")
multi_simlex.head()

Multi-SimLex (ES) — 1888 pares de palabras
Columnas de anotadores: ['Annotator 1', 'Annotator 2', 'Annotator 3', 'Annotator 4', 'Annotator 5', 'Annotator 6', 'Annotator 7', 'Annotator 8', 'Annotator 9', 'Annotator 10']


,PAIR_ID,SPA_W1,SPA_W2,Annotator 1,Annotator 2,Annotator 3,Annotator 4,Annotator 5,Annotator 6,Annotator 7,Annotator 8,Annotator 9,Annotator 10,SPA_AVG
0,1,brazo,músculo,1,1,5,0,0,0,2,1,2,2,1.4
1,2,democracia,monarquía,0,0,3,0,0,0,2,1,3,4,1.3
2,3,tejado,techo,5,5,6,4,5,3,4,4,6,6,4.8
3,4,amigo,profesor,0,0,1,0,0,0,2,0,1,0,0.4
4,5,mano,pie,1,0,3,0,0,0,2,1,3,1,1.1


In [14]:
# ver valores mínimo y máximo de las puntuaciones
min_score = multi_simlex['SPA_AVG'].min()
max_score = multi_simlex['SPA_AVG'].max()

print("Valor mínimo:", min_score)
print("Valor máximo:", max_score)

Valor mínimo: 0.0
Valor máximo: 5.9


## Columnas del dataset Spanish STS

| Columna | Descripción |
|---|---|
| **id** | Identificador único del par de frases. |
| **sentence1** | Primera frase del par. |
| **sentence2** | Segunda frase del par. |
| **score** | Puntuación de similitud semántica asignada por anotadores humanos. Cuanto mayor es el valor, más similares son las dos frases. |

In [15]:
# ============================================================
# DATASET 2 — Spanish STS (PlanTL-GOB-ES/sts-es)
# Avaluación extrínseca · Métrica: correlación de Pearson
# ============================================================
sts = load_dataset("PlanTL-GOB-ES/sts-es")

train_df = sts["train"].to_pandas().rename(columns={"label": "score"})
dev_df   = sts["validation"].to_pandas().rename(columns={"label": "score"})
test_df  = sts["test"].to_pandas().rename(columns={"label": "score"})

print(f"Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")
print(train_df.head())

Train: 1320 | Dev: 77 | Test: 155
  id                                          sentence1  \
0  0  Según el sondeo, 87% de los católicos cree que...   
1  1  La psicología explora conceptos como la percep...   
2  2  La tradición comenzó en el siglo IV, pero la m...   
3  3  "Maria Anna Schwegelin" (? - 1781 en la cárcel...   
4  4  Su identidad la había revelado durante el viaj...   

                                           sentence2  score  
0  El 87% de los católicos del mundo aprobaron el...   3.75  
1  La "psicología básica" es la parte de la psico...   2.80  
2  La tradición de erigir piedras con inscripcion...   2.40  
3  Te entregamos, Anna Schwegelin, al verdugo par...   2.20  
4  La información fue suministrada por el pescado...   2.20  


## Corpus de Wikipedia en español (`raw_es`)

Este corpus se utilizará para entrenar los modelos de embeddings (**Word2Vec** y **fastText**).

El corpus está dividido en múltiples archivos de texto dentro de la carpeta `raw_es`.

Cada archivo contiene una parte del texto extraído de Wikipedia en español.

En total se han encontrado **57 archivos** dentro del corpus.

In [16]:
# ============================================================
# DATASET 3 — Corpus Wikipedia en español
# ============================================================
corpus_dir   = "raw_es"
corpus_files = sorted(os.listdir(corpus_dir))

print(f"Total archivos encontrados: {len(corpus_files)}")
print(corpus_files[:5])

Total archivos encontrados: 57
['spanishText_10000_15000', 'spanishText_110000_115000', 'spanishText_120000_125000', 'spanishText_15000_20000', 'spanishText_180000_185000']


## 2. Preprocesamiento de *raw_es*

Es necesario realizar un preprocesamiento sobre el corpus *raw_es* para eliminar elementos 
que no aportan información: etiquetas **`<doc ...>`/`</doc>`**, **`ENDOFARTICLE`** y **URLs**.

También aplicaremos preprocesamiento básico: convertir a minúsculas, eliminar caracteres no 
alfabéticos y tokenizar.

Respecto a las **stopwords**, la literatura no es concluyente:

- La implementación de Gensim ya realiza *subsampling* automático de palabras frecuentes 
  (parámetro `sample=0.001`), por lo que las stopwords ya reciben menos peso durante el 
  entrenamiento sin necesidad de eliminarlas explícitamente.

- Por otro lado, las stopwords pueden ser útiles para asociar palabras relacionadas a través 
  del contexto sintáctico. Por ejemplo, nombres de ciudades pueden aparecer juntos no solo 
  por verbos como *"ir"* o *"viajar"*, sino también por preposiciones como *"a"*, *"desde"* 
  o *"en"* — relaciones que se perderían al eliminarlas.

- Además, el español tiene una morfología más rica que el inglés (artículos, preposiciones 
  contractas, pronombres clíticos...), por lo que eliminar stopwords puede suponer una 
  pérdida de contexto mayor que en otras lenguas.

Por este motivo, entrenaremos **dos modelos** para cada configuración (con y sin stopwords) 
y evaluaremos empíricamente cuál obtiene mejores resultados en Multi-SimLex y Spanish STS.

**Referencias**:
- Trideep Rath (2016). [*Stopword removing when using the word2vec*](https://stackoverflow.com/questions/34721984/stopword-removing-when-using-the-word2vec). Stack Overflow.
- Jindřich (2020). [*Text preprocessing for text classification using fastText*](https://stackoverflow.com/questions/62244474/text-preprocessing-for-text-classification-using-fasttext). Stack Overflow.

In [17]:
import unicodedata
import re
import os
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    texto = unicodedata.normalize("NFKC", texto)
    texto = texto.lower()
    texto = re.sub(r'https?://\S+|www\.\S+', '', texto)
    texto = re.sub(r'<[^>]+>', ' ', texto)
    texto = re.sub(r'endofarticle\.?', ' ', texto, flags=re.IGNORECASE)
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def preprocesar_linea(linea, eliminar_stopwords=True):
    linea = linea.strip()
    if not linea:
        return []
    texto = limpiar_texto(linea)
    palabras = texto.split()
    if eliminar_stopwords:
        palabras = [p for p in palabras if p and p not in STOPWORDS]
    else:
        palabras = [p for p in palabras if p]
    return palabras

class CorpusFactory:
    """
    Iterador re-entrante sobre el corpus raw_es.
    Gensim necesita dos pasadas (build_vocab + train),
    por eso cada llamada a __iter__ genera un iterador nuevo.
    """
    def __init__(self, directorio, archivos, eliminar_stopwords=True):
        self.directorio = directorio
        self.archivos = archivos
        self.eliminar_stopwords = eliminar_stopwords

    def __iter__(self):
        for nombre_archivo in self.archivos:
            ruta = os.path.join(self.directorio, nombre_archivo)
            with open(ruta, "r", encoding="latin-1", errors="ignore") as f:
                for linea in f:
                    palabras = preprocesar_linea(linea, self.eliminar_stopwords)
                    if palabras:
                        yield palabras

In [18]:
corpus_con_sw = CorpusFactory(corpus_dir, corpus_files, eliminar_stopwords=False)
corpus_sin_sw = CorpusFactory(corpus_dir, corpus_files, eliminar_stopwords=True)

N = 20  # Cambia este número para ver más o menos frases

# Verificación visual
print("-- CON stopwords --")
for i, frase in enumerate(corpus_con_sw):
    print(f"  [{i+1}] {frase[:10]}")
    if i == N - 1: break

print("\n-- SIN stopwords --")
for i, frase in enumerate(corpus_sin_sw):
    print(f"  [{i+1}] {frase[:10]}")
    if i == N - 1: break

-- CON stopwords --
  [1] ['acontecimientos']
  [2] ['nacimientos']
  [3] ['fallecimientos']
  [4] ['fulgencio', 'de', 'écija', 'santo', 'español']
  [5] ['erquinoaldo', 'mayordomo', 'franco', 'de', 'palacio', 'de', 'neustria']
  [6] ['acontecimientos']
  [7] ['nacimientos']
  [8] ['egilona', 'última', 'reina', 'visigoda', 'de', 'hispania']
  [9] ['fallecimientos']
  [10] ['acontecimientos']
  [11] ['fin', 'del', 'califato', 'perfecto', 'los', 'omeyas', 'en', 'el', 'poder', 'califato']
  [12] ['nacimientos']
  [13] ['fallecimientos']
  [14] ['acontecimientos']
  [15] ['los', 'omeyas', 'acceden', 'al', 'califato']
  [16] ['nacimientos']
  [17] ['fallecimientos']
  [18] ['mundo', 'islámico', 'alí', 'murió', 'traicionado', 'por', 'los', 'suyos', 'y', 'asesinado']
  [19] ['acontecimientos']
  [20] ['nacimientos']

-- SIN stopwords --
  [1] ['acontecimientos']
  [2] ['nacimientos']
  [3] ['fallecimientos']
  [4] ['fulgencio', 'écija', 'santo', 'español']
  [5] ['erquinoaldo', 'mayordomo', '

## 3. Funciones de Evaluación

### Evaluación intrínseca (Multi-SimLex)

Para cada par de palabras, obtenemos sus vectores y calculamos la **similitud coseno**. Posteriormente, calculamos la **correlación de Spearman** entre las similitudes predichas y las puntuaciones humanas.

Tal como se remarcó en las sesiones de laboratorio, es interesante evaluar la **puntuación de cada anotador** individualmente y realizar el promedio, lo que refleja mejor la variabilidad humana frente a una única media agregada.

Las palabras fuera del vocabulario (OOV, *Out-Of-Vocabulary*) se descartan del cálculo de la correlación. En este aspecto, **fastText** ofrece una ventaja competitiva al gestionar las OOV mediante el uso de n-gramas de caracteres.

In [19]:
def cosine_sim(v1, v2):
    """Similitud coseno entre dos vectores."""
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (n1 * n2))

def get_vector(model, word):
    """Obtiene el vector de una palabra (compatible con Word2Vec y FastText de gensim)."""
    try:
        return model.wv[word]
    except KeyError:
        return None

def intrinsic_eval(model, df, annotator_cols):
    """
    Evaluación intrínseca sobre Multi-SimLex.
    Retorna:
      - spearman_avg: correlación de Spearman respecto a SPA_AVG
      - spearman_annotators: media de las correlaciones por anotador
      - oov_rate: proporción de pares con palabras fuera del vocabulario (OOV)
    """
    sims, golds_avg, golds_annot = [], [], [[] for _ in annotator_cols]
    oov_count = 0

    for _, row in df.iterrows():
        v1 = get_vector(model, row['SPA_W1'])
        v2 = get_vector(model, row['SPA_W2'])
        if v1 is None or v2 is None:
            oov_count += 1
            continue
        sims.append(cosine_sim(v1, v2))
        golds_avg.append(row['SPA_AVG'])
        for i, col in enumerate(annotator_cols):
            golds_annot[i].append(row[col])

    if len(sims) < 2:
        return 0.0, 0.0, 1.0

    spearman_avg = spearmanr(sims, golds_avg).statistic

    # Correlación por anotador (puntuación de cada anotador individualmente)
    corrs_annot = [spearmanr(sims, g).statistic for g in golds_annot if len(g) > 1]
    spearman_annotators = float(np.mean(corrs_annot))

    oov_rate = oov_count / len(df)
    return spearman_avg, spearman_annotators, oov_rate

def truncar_vectores(model, new_dim):
    """
    Trunca los vectores del model a 'new_dim' dimensiones.
    """
    from gensim.models import KeyedVectors
    kv_trunc = KeyedVectors(vector_size=new_dim)
    words = list(model.wv.index_to_key)
    vectors = model.wv.vectors[:, :new_dim]
    kv_trunc.add_vectors(words, vectors)
    
    # Wrapper para mantener la interfaz .wv
    class ModelWrapper:
        def __init__(self, kv): self.wv = kv
    return ModelWrapper(kv_trunc)

print("Funciones de evaluación intrínseca definidas ✓")

Funciones de evaluación intrínseca definidas ✓


## 4. Word2Vec: CBOW y Skip-gram

Entrenamos modelos Word2Vec con ambas arquitecturas. Tal como explica la teoría:
- **CBOW** (*Continuous Bag-of-Words*): predice la palabra central a partir de las palabras de contexto.
- **Skip-gram**: predice las palabras de contexto a partir de la palabra central.

Experimentamos con:
1. Presencia/ausencia de stopwords.
2. Dimensiones del vector (100 vs 300) + truncado a 100d.
3. Cantidad de corpus (efecto del tamaño del corpus).

> **Nota de implementación**: Nunca debemos usar el modelo de Word2Vec directamente para pasarlo a la capa de embeddings de la red neuronal sin restringir el vocabulario. Es necesario construir una matriz de embeddings específica con el vocabulario del corpus de entrenamiento para optimizar la memoria y el rendimiento.

In [ ]:
# ── Grid Search Word2Vec ─────────────────────────────────────────────

param_grid = {
    'vector_size': [100, 300],
    'window':      [3, 5],
    'min_count':   [5, 10],
    'epochs':      [5, 10],
}

claves        = list(param_grid.keys())
combinaciones = list(product(*param_grid.values()))

# Configuración de archivos (usa las variables que definiste al principio)
N_ARCHIVOS_MUESTRA    = 5    # Para el grid search rápido
N_ARCHIVOS_FINAL      = 20   # Para ampliar el mejor modelo
# Asegúrate de que corpus_dir y corpus_files estén definidos previamente
EXPERIMENTO_N_ARCHIVOS = [5, 10, 20, len(corpus_files)] if 'corpus_files' in locals() else []

resultados_globales = []
modelos_w2v = {}  # Guardamos los mejores modelos para uso posterior

for sg, nombre_arq in [(0, 'CBOW'), (1, 'Skip-gram')]:
    for eliminar_sw, sufijo_sw in [(True, 'sin_sw'), (False, 'con_sw')]:
        nombre_modelo = f"w2v_{nombre_arq.lower().replace('-','_')}_{sufijo_sw}.model"
        print(f"\n{'='*60}")
        print(f"  {nombre_arq} — {sufijo_sw.replace('_',' ')}")
        print(f"{'='*60}")

        if os.path.exists(nombre_modelo):
            print(f"  Cargando modelo existente: {nombre_modelo}...")
            mejor_modelo = Word2Vec.load(nombre_modelo)
            print(f"  Cargado ✓ — vocabulario: {len(mejor_modelo.wv):,} palabras")
        elif 'corpus_files' in locals():
            # Grid search sobre muestra pequeña
            print(f"  Grid search ({len(combinaciones)} combinaciones)...")
            # ¡OJO! Aquí usamos corpus_dir en minúsculas como lo tenías antes
            corpus_muestra = list(
                CorpusFactory(corpus_dir, corpus_files[:N_ARCHIVOS_MUESTRA], 
                              eliminar_stopwords=eliminar_sw)
            )

            resultados, mejor_score, mejor_modelo = [], -np.inf, None
            for i, combo in enumerate(combinaciones):
                params = dict(zip(claves, combo))
                modelo = Word2Vec(
                    sentences=corpus_muestra,
                    vector_size=params['vector_size'],
                    window=params['window'],
                    min_count=params['min_count'],
                    sg=sg, workers=4, epochs=params['epochs'], seed=SEED
                )
                sp_avg, sp_ann, oov = intrinsic_eval(modelo, multi_simlex, annotator_cols)
                resultados.append({**params, 'spearman_avg': sp_avg, 'spearman_ann': sp_ann, 'oov': oov})
                print(f"  [{i+1}/{len(combinaciones)}] {params} → Spearman avg: {sp_avg:.4f}")
                if sp_avg > mejor_score:
                    mejor_score, mejor_modelo = sp_avg, modelo

            df_res = pd.DataFrame(resultados).sort_values('spearman_avg', ascending=False)
            print(f"\n  Mejores hiperparámetros:")
            print(df_res.iloc[0].to_string())

            # Ampliar con más datos
            print(f"\n  Ampliando a {N_ARCHIVOS_FINAL} archivos...")
            corpus_final = list(
                CorpusFactory(corpus_dir, corpus_files[:N_ARCHIVOS_FINAL], 
                              eliminar_stopwords=eliminar_sw)
            )
            mejor_modelo.build_vocab(corpus_final, update=True)
            mejor_modelo.train(corpus_final, total_examples=len(corpus_final), 
                               epochs=mejor_modelo.epochs)
            mejor_modelo.save(nombre_modelo)
            print(f"  Modelo guardado como '{nombre_modelo}' ✓")
        else:
            print("  AVISO: Corpus no disponible. Saltando..."); continue

        # Evaluación final
        sp_avg, sp_ann, oov = intrinsic_eval(mejor_modelo, multi_simlex, annotator_cols)
        print(f"  Spearman (SPA_AVG): {sp_avg:.4f}")
        print(f"  Spearman (por anotador, media): {sp_ann:.4f}")
        print(f"  OOV rate: {oov:.2%}")

        # Experimento: truncado de vectores a 100d
        if mejor_modelo.vector_size > 100:
            m_trunc = truncar_vectores(mejor_modelo, 100)
            sp_trunc, _, _ = intrinsic_eval(m_trunc, multi_simlex, annotator_cols)
            print(f"  Truncado a 100d → Spearman: {sp_trunc:.4f}")

        # Experimento: efecto tamaño del corpus
        for n_arch in EXPERIMENTO_N_ARCHIVOS:
            n_arch = min(n_arch, len(corpus_files))
            corp_exp = list(CorpusFactory(corpus_dir, corpus_files[:n_arch], 
                                          eliminar_stopwords=eliminar_sw))
            m_exp = Word2Vec(
                sentences=corp_exp,
                vector_size=mejor_modelo.vector_size,
                window=mejor_modelo.window,
                min_count=mejor_modelo.min_count,
                sg=sg, workers=4, epochs=mejor_modelo.epochs, seed=SEED
            )
            sp_e, _, _ = intrinsic_eval(m_exp, multi_simlex, annotator_cols)
            resultados_globales.append({'arquitectura': nombre_arq, 
                                        'stopwords': sufijo_sw, 
                                        'n_archivos': n_arch, 
                                        'spearman': sp_e})

        modelos_w2v[f"{nombre_arq}_{sufijo_sw}"] = mejor_modelo

print("\nWord2Vec completado")


  CBOW — sin sw
  Grid search (16 combinaciones)...
  [1/16] {'vector_size': 100, 'window': 3, 'min_count': 5, 'epochs': 5} → Spearman avg: 0.2610
  [2/16] {'vector_size': 100, 'window': 3, 'min_count': 5, 'epochs': 10} → Spearman avg: 0.3111
  [3/16] {'vector_size': 100, 'window': 3, 'min_count': 10, 'epochs': 5} → Spearman avg: 0.2680
  [4/16] {'vector_size': 100, 'window': 3, 'min_count': 10, 'epochs': 10} → Spearman avg: 0.3156
  [5/16] {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 5} → Spearman avg: 0.2842
  [6/16] {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 10} → Spearman avg: 0.3209
  [7/16] {'vector_size': 100, 'window': 5, 'min_count': 10, 'epochs': 5} → Spearman avg: 0.2798
  [8/16] {'vector_size': 100, 'window': 5, 'min_count': 10, 'epochs': 10} → Spearman avg: 0.3239


In [ ]:
# ── Visualización: efecto del tamaño del corpus ──────────────────────
if resultados_globales:
    df_exp = pd.DataFrame(resultados_globales)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Efecto del tamaño del corpus — Word2Vec', fontsize=13)
    
    for ax, sw in zip(axes, ['sin_sw', 'con_sw']):
        subset = df_exp[df_exp['stopwords'] == sw]
        for arq, color in [('CBOW', '#2196F3'), ('Skip-gram', '#FF9800')]:
            d = subset[subset['arquitectura'] == arq].sort_values('n_archivos')
            ax.plot(d['n_archivos'], d['spearman'], 'o-', label=arq,
                    color=color, linewidth=2, markersize=7)
        
        ax.set_title(sw.replace('_', ' ').title().replace('Sin Sw', 'Sin Stopwords').replace('Con Sw', 'Con Stopwords'))
        ax.set_xlabel('Número de archivos del corpus')
        ax.set_ylabel('Spearman r (Multi-SimLex)')
        ax.legend()
        ax.grid(alpha=0.3)
        
    plt.tight_layout()
    plt.savefig('efecto_corpus.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. fastText

fastText (Facebook, 2016) es una mejora de Word2Vec que representa cada palabra como la **suma de los embeddings de sus n-gramas de caracteres** (por defecto $3 \le n \le 6$). Esto permite:
- Capturar información morfológica.
- **Gestionar palabras OOV** (*Out-of-Vocabulary*): incluso para palabras no vistas durante el entrenamiento, se puede construir un vector a partir de sus n-gramas.

Comparamos dos variantes:
1. **fastText entrenado por nosotros** sobre el Wikicorpus en español.
2. **fastText oficial** (Facebook, pre-entrenado sobre Common Crawl): `cc.es.300.bin`.

> **Nota sobre OOV**: El profesor remarcó que la puntuación de fastText para palabras OOV puede ser inferior que para palabras presentes en el corpus, ya que los n-gramas no siempre capturan todo el significado semántico. No obstante, el fastText oficial, al haber sido entrenado con un volumen de texto mucho mayor, tiende a ofrecer una cobertura superior.

In [ ]:
# ── fastText entrenado sobre Wikicorpus ──────────────────────────────
FT_MODEL_PATH = 'ft_wiki_es.model'

# Asegúrate de haber importado: from gensim.models import FastText as GensimFastText
if os.path.exists(FT_MODEL_PATH):
    print("Cargando modelo fastText existente...")
    ft_wiki = GensimFastText.load(FT_MODEL_PATH)
    print(f"Cargado ✓ — vocabulario: {len(ft_wiki.wv):,} palabras")
elif 'corpus_files' in locals() and corpus_files:
    print("Entrenando fastText sobre Wikicorpus (esto puede tardar)...")
    # Usamos la misma cantidad de archivos que para el modelo final de Word2Vec
    corpus_ft = list(CorpusFactory(corpus_dir, corpus_files[:N_ARCHIVOS_FINAL]))
    
    ft_wiki = GensimFastText(
        sentences=corpus_ft,
        vector_size=300,
        window=5,
        min_count=5,
        workers=4,
        epochs=10,
        min_n=3, max_n=6,  # Rango de n-gramas de caracteres (Teoría Lab 10, pág 9)
        seed=SEED
    )
    ft_wiki.save(FT_MODEL_PATH)
    print(f"fastText wiki guardado ✓")

    # Evaluación intrínseca
    sp_avg, sp_ann, oov = intrinsic_eval(ft_wiki, multi_simlex, annotator_cols)
    print(f"fastText Wiki — Spearman avg: {sp_avg:.4f} | por anotador: {sp_ann:.4f} | OOV: {oov:.2%}")
else:
    print("Corpus no disponible, saltando entrenamiento de fastText Wiki.")
    ft_wiki = None

In [ ]:
# ── fastText oficial (Facebook) ─────────────────────────────────────
# Descarga: https://fasttext.cc/docs/en/crawl-vectors.html
# wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.bin.gz
# gunzip cc.es.300.bin.gz

FT_OFICIAL_PATH = 'cc.es.300.bin'

if os.path.exists(FT_OFICIAL_PATH):
    import gensim
    print("Cargando fastText oficial (puede tardar unos minutos)...")
    # load_facebook_model es necesario para leer el formato .bin original de Facebook
    ft_oficial = gensim.models.fasttext.load_facebook_model(FT_OFICIAL_PATH)
    
    # Evaluación intrínseca
    sp_avg_of, sp_ann_of, oov_of = intrinsic_eval(ft_oficial, multi_simlex, annotator_cols)
    
    print(f"fastText Oficial — Spearman avg: {sp_avg_of:.4f} | por anotador: {sp_ann_of:.4f} | OOV: {oov_of:.2%}")
    print("\nOOV: fastText oficial gestiona OOV mediante n-gramas.")
    print("Ejemplo: 'palabrarara' → vector construido a partir de sus 3-6-gramas.")
else:
    print(f"Modelo oficial no encontrado en '{FT_OFICIAL_PATH}'.")
    print("Descárgalo de: https://fasttext.cc/docs/en/crawl-vectors.html")
    ft_oficial = None

## 6. Resumen de Evaluación Intrínseca (Multi-SimLex)

Recopilamos todos los resultados obtenidos en la evaluación intrínseca. Un coeficiente de Spearman más alto indica que el modelo es capaz de capturar con mayor fidelidad la similitud semántica tal como la perciben los humanos.

En esta comparativa final analizamos:
1. El impacto de la **arquitectura** (Word2Vec vs. fastText).
2. El efecto del **volumen de datos** (corpus propio vs. oficial).
3. La importancia del **preprocesamiento** (presencia de stopwords).

In [ ]:
resumen_intrinseco = []

for nom, model in modelos_w2v.items():
    sp_avg, sp_ann, oov = intrinsic_eval(model, multi_simlex, annotator_cols)
    resumen_intrinseco.append({
        'Modelo': f'Word2Vec-{nom}',
        'Dimensión': model.vector_size,
        'Spearman (avg)': round(sp_avg, 4),
        'Spearman (por anot.)': round(sp_ann, 4),
        'OOV (%)': round(oov * 100, 2)
    })

if ft_wiki is not None:
    sp_avg, sp_ann, oov = intrinsic_eval(ft_wiki, multi_simlex, annotator_cols)
    resumen_intrinseco.append({
        'Modelo': 'fastText-Wiki', 
        'Dimensión': ft_wiki.vector_size,
        'Spearman (avg)': round(sp_avg, 4),
        'Spearman (por anot.)': round(sp_ann, 4),
        'OOV (%)': round(oov * 100, 2)
    })

if ft_oficial is not None:
    sp_avg, sp_ann, oov = intrinsic_eval(ft_oficial, multi_simlex, annotator_cols)
    resumen_intrinseco.append({
        'Modelo': 'fastText-Oficial', 
        'Dimensión': 300,
        'Spearman (avg)': round(sp_avg, 4),
        'Spearman (por anot.)': round(sp_ann, 4),
        'OOV (%)': round(oov * 100, 2)
    })

df_resumen = pd.DataFrame(resumen_intrinseco).sort_values('Spearman (avg)', ascending=False)

print('=== Resumen Evaluación Intrínseca (Multi-SimLex ES) ===')
print(df_resumen.to_string(index=False))

## 7. Evaluación Extrínseca — Baseline Coseno

El modelo baseline representa cada frase como un **vector agregado** de sus embeddings y calcula la **similitud coseno** entre las representaciones de las dos frases. Este enfoque no requiere entrenamiento adicional (es *unsupervised*).

Comparamos dos estrategias de agregación:
1. **Media simple**: `mean(vectores de las palabras de la frase)`
2. **Media ponderada con TF-IDF**: $V_d = \frac{\sum TF-IDF(t,d,D) \cdot V_t}{\sum TF-IDF(t,d,D)}$

La ponderación TF-IDF reduce el peso de las palabras muy frecuentes (como las stop-words funcionales) que aportan poco significado semántico, priorizando los términos que realmente definen el contenido de la frase.

In [ ]:
def sentence_vector_mean(sentence, model_wv):
    """Representación de frase como media simple de los embeddings de las palabras."""
    # Asegúrate de que la función se llame 'limpiar_texto' o 'preprocesar_linea' según tu código inicial
    tokens = preprocesar_linea(sentence, eliminar_stopwords=False)
    vecs = [model_wv[t] for t in tokens if t in model_wv]
    if not vecs:
        return np.zeros(model_wv.vector_size)
    return np.mean(vecs, axis=0)

def build_tfidf(sentences):
    """Construye la matriz TF-IDF sobre un conjunto de frases."""
    vec = TfidfVectorizer(use_idf=True, smooth_idf=True, norm=None)
    matrix = vec.fit_transform(sentences)
    feature_names = np.array(vec.get_feature_names_out())
    return vec, matrix, feature_names

def sentence_vector_tfidf(sentence, tfidf_row, feature_names, model_wv):
    """
    Representación de frase como media ponderada de los embeddings
    con pesos TF-IDF. Las palabras sin embedding se descartan.
    """
    indices = tfidf_row.indices
    scores  = tfidf_row.data
    weighted_sum = np.zeros(model_wv.vector_size, dtype=np.float32)
    total_weight = 0.0
    for idx, score in zip(indices, scores):
        word = feature_names[idx]
        if word in model_wv:
            weighted_sum += score * model_wv[word]
            total_weight += score
    if total_weight == 0:
        return sentence_vector_mean(sentence, model_wv)
    return weighted_sum / total_weight

def evaluate_baseline_simple(df, model_wv):
    """Versión simplificada y rápida del baseline: media simple y tfidf."""
    # Preparar TF-IDF sobre todas las frases del split (concatenamos s1 y s2)
    all_sents = [str(r) for row in df[['sentence1','sentence2']].values for r in row]
    tfidf_vec = TfidfVectorizer(use_idf=True, smooth_idf=True, norm=None)
    tfidf_mat = tfidf_vec.fit_transform(all_sents)
    feat_names = np.array(tfidf_vec.get_feature_names_out())

    preds_mean, preds_tfidf, golds = [], [], []
    for i, row in enumerate(df.itertuples()):
        s1, s2 = str(row.sentence1), str(row.sentence2)
        
        # Obtenemos los vectores (el índice en tfidf_mat es i*2 para s1 e i*2+1 para s2)
        v1_m = sentence_vector_mean(s1, model_wv)
        v2_m = sentence_vector_mean(s2, model_wv)
        v1_t = sentence_vector_tfidf(s1, tfidf_mat[i*2],   feat_names, model_wv)
        v2_t = sentence_vector_tfidf(s2, tfidf_mat[i*2+1], feat_names, model_wv)
        
        preds_mean.append(cosine_sim(v1_m, v2_m))
        preds_tfidf.append(cosine_sim(v1_t, v2_t))
        golds.append(row.score)

    # Calculamos Pearson (Estándar para STS según Lab 10)
    pr_mean  = pearsonr(preds_mean,  golds).statistic
    pr_tfidf = pearsonr(preds_tfidf, golds).statistic
    return pr_mean, pr_tfidf

# ── Ejecución de la Evaluación Extrínseca ──────────────────────────────
print('=== Baseline Coseno (Evaluación Extrínseca STS) ===')
resumen_extrinseco = []

# Nota: Asegúrate de que 'test_df' es tu dataframe de STS
for nom, model in modelos_w2v.items():
    pr_mean, pr_tfidf = evaluate_baseline_simple(test_df, model.wv)
    print(f"Word2Vec-{nom} | Media simple: {pr_mean:.4f} | TF-IDF: {pr_tfidf:.4f}")
    resumen_extrinseco.append({'Modelo': f'Baseline-Mean (W2V-{nom})', 'Pearson (test)': round(pr_mean, 4)})
    resumen_extrinseco.append({'Modelo': f'Baseline-TFIDF (W2V-{nom})', 'Pearson (test)': round(pr_tfidf, 4)})

if ft_oficial is not None:
    pr_mean, pr_tfidf = evaluate_baseline_simple(test_df, ft_oficial.wv)
    print(f"fastText-Oficial | Media simple: {pr_mean:.4f} | TF-IDF: {pr_tfidf:.4f}")
    resumen_extrinseco.append({'Modelo': 'Baseline-Mean (FT-Oficial)', 'Pearson (test)': round(pr_mean, 4)})
    resumen_extrinseco.append({'Modelo': 'Baseline-TFIDF (FT-Oficial)', 'Pearson (test)': round(pr_tfidf, 4)})

## 8. Model Secuencial Siamés (con Embeddings Estáticos)

Damos un paso más allá del baseline y entrenamos un modelo neuronal que procesa las frases como **secuencias** de índices de palabras.

### Arquitectura
```
Input: secuencia de índices de palabras
    → Embedding (pre-entrenado Word2Vec/fastText, o aleatorio)
    → BiLSTM (bidireccional)
    → Attention Pooling
    → [h1, h2, |h1-h2|, h1*h2]  (concatenación de características)
    → MLP de regresión
    → valor de similitud (0 a 5)
```

### Consideraciones importantes (sesión de laboratorio)
- **Restricción del Vocabulario**: Tal como remarcó el profesor, nunca se debe pasar el modelo Word2Vec completo a la capa de embedding. Es fundamental construir una **matriz de embeddings restringida** únicamente al vocabulario presente en nuestro corpus de entrenamiento (STS). Esto optimiza drásticamente el uso de memoria.

- **Congelación de pesos**: Debemos comparar el uso de **embeddings congelados** (`freeze=True`) frente a **entrenables** (`freeze=False`). Los pre-entrenados actúan como un excelente punto de partida (*transfer learning*) que acelera la convergencia.

- **Riesgo de Overfitting**: Si utilizamos embeddings aleatorios con un corpus pequeño, el modelo tenderá al **sobreajuste**. Es vital dimensionar la red adecuadamente y usar regularización.

In [ ]:
# ── Preparación: vocabulario y matriz de embeddings ─────────────────────

def build_vocab_and_matrix(sentences, model_wv, max_vocab=50000):
    """
    Construye el vocabulario y la matriz de embeddings a partir
    de las frases del corpus STS y el modelo de word embeddings.

    IMPORTANTE: Restringimos el vocabulario a las palabras presentes
    en el corpus de entrenamiento (Recomendación: 'Restringir el vocabulario
    solo a lo que existe en nuestro dataset para optimizar memoria').
    """
    PAD_IDX, UNK_IDX = 0, 1
    word2idx = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
    dim = model_wv.vector_size

    # Recoger las palabras que aparecen en el corpus STS
    vocab_counter = {}
    for sent in sentences:
        # Nota: Asegúrate de usar tu función 'preprocesar_linea' o 'limpiar_texto'
        for w in preprocesar_linea(str(sent), eliminar_stopwords=False):
            vocab_counter[w] = vocab_counter.get(w, 0) + 1

    # Construir la matriz: PAD (ceros) + UNK (aleatorio) + palabras del vocab
    matrix = [np.zeros(dim), np.random.randn(dim) * 0.01]
    
    # Ordenamos por frecuencia y filtramos para quedarnos con lo que el modelo conoce
    for w, _ in sorted(vocab_counter.items(), key=lambda x: -x[1])[:max_vocab]:
        if w in model_wv:
            word2idx[w] = len(matrix)
            matrix.append(model_wv[word2idx[w]]) # Acceso al vector del modelo
            
    matrix = np.array(matrix, dtype=np.float32)
    coverage = sum(1 for w in vocab_counter if w in model_wv) / max(len(vocab_counter), 1)
    
    print(f"   Vocabulario: {len(word2idx):,} palabras | Cobertura de embeddings: {coverage:.1%}")
    return word2idx, matrix, PAD_IDX, UNK_IDX

def tokenize_sentences(sentences, word2idx, unk_idx, max_len=64):
    """Convierte frases a secuencias de índices con truncamiento."""
    result = []
    for sent in sentences:
        ids = [word2idx.get(w, unk_idx) for w in preprocesar_linea(str(sent), eliminar_stopwords=False)][:max_len]
        result.append(ids)
    return result

def pad_sequences(seqs, pad_idx=0, max_len=64):
    """Añade padding para que todas las secuencias tengan la misma longitud."""
    max_l = min(max(len(s) for s in seqs), max_len)
    padded = np.full((len(seqs), max_l), pad_idx, dtype=np.int64)
    masks  = np.zeros((len(seqs), max_l), dtype=bool)
    for i, s in enumerate(seqs):
        l = min(len(s), max_l)
        padded[i, :l] = s[:l]
        masks[i,  :l] = True
    return padded, masks

class STSDataset(Dataset):
    """Dataset de PyTorch para cargar pares de frases STS."""
    def __init__(self, df, word2idx, unk_idx, max_len=64):
        s1 = tokenize_sentences(df['sentence1'].tolist(), word2idx, unk_idx, max_len)
        s2 = tokenize_sentences(df['sentence2'].tolist(), word2idx, unk_idx, max_len)
        self.ids1, self.masks1 = pad_sequences(s1, max_len=max_len)
        self.ids2, self.masks2 = pad_sequences(s2, max_len=max_len)
        self.scores = df['score'].values.astype(np.float32)

    def __len__(self): return len(self.scores)
    
    def __getitem__(self, i):
        return (torch.tensor(self.ids1[i]), torch.tensor(self.masks1[i]),
                torch.tensor(self.ids2[i]), torch.tensor(self.masks2[i]),
                torch.tensor(self.scores[i]))

print("Funciones de preparación de datos definidas ✓")

In [ ]:
# ── Arquitectura del Modelo Siamés ────────────────────────────────────

class AttentionPooling(nn.Module):
    """Pooling con atención: pondera los hidden states de la BiLSTM según su importancia."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x, mask):
        # x: [batch, seq_len, hidden_dim * 2]
        scores = self.proj(x).squeeze(-1)
        # Aplicamos la máscara para ignorar los tokens de <PAD>
        scores = scores.masked_fill(~mask, -1e9)
        alpha  = torch.softmax(scores, dim=-1)
        # Retornamos la suma ponderada de los estados ocultos
        return torch.sum(x * alpha.unsqueeze(-1), dim=1)

class SiameseBiLSTMAttention(nn.Module):
    """
    Modelo Siamés con BiLSTM y Attention Pooling.
    Arquitectura: Embedding → BiLSTM → Attention → [h1, h2, |h1-h2|, h1*h2] → MLP
    """
    def __init__(self, embedding_matrix, hidden_size=64, final_hidden_size=32,
                  trainable_embeddings=False):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float),
            freeze=not trainable_embeddings,   # freeze=True → embeddings congelados
            padding_idx=0
        )
        emb_dim = embedding_matrix.shape[1]
        # BiLSTM para capturar el contexto secuencial
        self.encoder = nn.LSTM(emb_dim, hidden_size, batch_first=True, bidirectional=True)
        self.pool     = AttentionPooling(hidden_size * 2)
        
        # El regressor recibe la concatenación de 4 vectores (h1, h2, diff, prod)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size * 2 * 4, final_hidden_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(final_hidden_size, 1)
        )

    def encode(self, input_ids, attention_mask):
        x = self.embedding(input_ids)
        x, _ = self.encoder(x)
        return self.pool(x, attention_mask)

    def forward(self, ids1, mask1, ids2, mask2):
        h1 = self.encode(ids1, mask1)
        h2 = self.encode(ids2, mask2)
        # Combinación heurística de vectores (pág. 27 Lab 10)
        feats = torch.cat([h1, h2, (h1 - h2).abs(), h1 * h2], dim=-1)
        return self.regressor(feats).squeeze(-1)


def train_siamese(model, train_dl, dev_dl, epochs=10, lr=1e-3, patience=3):
    """Bucle de entrenamiento con early stopping basado en la correlación de Pearson."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss() # Tarea de regresión (0 a 5)
    best_pearson, best_state, wait = -1, None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for ids1, mask1, ids2, mask2, scores in train_dl:
            ids1, mask1, ids2, mask2, scores = (
                ids1.to(DEVICE), mask1.to(DEVICE),
                ids2.to(DEVICE), mask2.to(DEVICE), scores.to(DEVICE)
            )
            opt.zero_grad()
            preds = model(ids1, mask1, ids2, mask2)
            loss = criterion(preds, scores)
            loss.backward()
            opt.step()
            total_loss += loss.item()

        # Validación en cada época
        model.eval()
        all_preds, all_golds = [], []
        with torch.no_grad():
            for ids1, mask1, ids2, mask2, scores in dev_dl:
                p = model(ids1.to(DEVICE), mask1.to(DEVICE),
                          ids2.to(DEVICE), mask2.to(DEVICE)).cpu().numpy()
                all_preds.extend(p)
                all_golds.extend(scores.numpy())
        
        # Calculamos Pearson (métrica clave de STS)
        pearson = pearsonr(all_preds, all_golds).statistic
        print(f"  Época {epoch:2d} | Loss: {total_loss/len(train_dl):.4f} | Pearson dev: {pearson:.4f}")

        # Guardar el mejor modelo (Early Stopping)
        if pearson > best_pearson:
            best_pearson, best_state, wait = pearson, model.state_dict().copy(), 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping en la época {epoch}")
                break

    model.load_state_dict(best_state)
    return best_pearson

print("Arquitectura Siamés definida ✓")

In [ ]:
# ── Entrenamiento: Modelo Siamés con Word2Vec ───────────────────────────
# Usamos el mejor modelo Word2Vec disponible de los entrenados previamente

def eval_pearson(model, dl):
    model.eval()
    preds, golds = [], []
    with torch.no_grad():
        for ids1, mask1, ids2, mask2, scores in dl:
            ids1, mask1 = ids1.to(DEVICE), mask1.to(DEVICE)
            ids2, mask2 = ids2.to(DEVICE), mask2.to(DEVICE)
            # El modelo devuelve el coseno predicho
            p = model(ids1, mask1, ids2, mask2).cpu().numpy()
            preds.extend(p)
            golds.extend(scores.numpy())
    
    # pearsonr devuelve (coeficiente, p-value), nos quedamos con el coeficiente [0]
    return pearsonr(preds, golds)[0]


BASE_MODEL_KEY = next(iter(modelos_w2v)) if modelos_w2v else None

if BASE_MODEL_KEY is not None:
    base_model = modelos_w2v[BASE_MODEL_KEY]
    print(f"Modelo base: Word2Vec-{BASE_MODEL_KEY} ({base_model.vector_size}d)")

    # Preparar vocabulario y matriz de embeddings
    # Unimos todas las frases del dataset STS para construir el vocabulario restringido
    all_sents = (train_df['sentence1'].tolist() + train_df['sentence2'].tolist() +
                 dev_df['sentence1'].tolist()   + dev_df['sentence2'].tolist())
    
    print("Construyendo vocabulario y matriz de embeddings...")
    word2idx, emb_matrix, PAD_IDX, UNK_IDX = build_vocab_and_matrix(all_sents, base_model.wv)

    # Datasets y DataLoaders de PyTorch
    train_ds = STSDataset(train_df, word2idx, UNK_IDX)
    dev_ds   = STSDataset(dev_df,   word2idx, UNK_IDX)
    test_ds  = STSDataset(test_df,  word2idx, UNK_IDX)
    
    train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
    dev_dl   = DataLoader(dev_ds,   batch_size=64)
    test_dl  = DataLoader(test_ds,  batch_size=64)

    # Experimento: Embeddings Pre-entrenados (Congelados vs Entrenables)
    for trainable, label in [(False, 'congelado'), (True, 'entrenable')]:
        print(f"\n--- Embedding {label} ---")
        model_siam = SiameseBiLSTMAttention(
            emb_matrix, hidden_size=64, final_hidden_size=32,
            trainable_embeddings=trainable
        ).to(DEVICE)
        
        best_pear = train_siamese(model_siam, train_dl, dev_dl, epochs=15, lr=1e-3, patience=3)
        test_pear = eval_pearson(model_siam, test_dl)
        
        print(f"   Pearson test: {test_pear:.4f}")
        resumen_extrinseco.append({
            'Modelo': f'Siamés-BiLSTM (W2V, emb {label})',
            'Pearson (test)': round(test_pear, 4)
        })

    # Experimento: Embedding aleatorio (sin conocimiento previo)
    print("\n--- Embedding aleatorio ---")
    rand_matrix = np.random.randn(len(word2idx), base_model.vector_size).astype(np.float32) * 0.01
    model_rand = SiameseBiLSTMAttention(
        rand_matrix, hidden_size=64, final_hidden_size=32, trainable_embeddings=True
    ).to(DEVICE)
    
    best_pear = train_siamese(model_rand, train_dl, dev_dl, epochs=15, lr=1e-3, patience=3)
    test_pear_rand = eval_pearson(model_rand, test_dl)
    
    print(f"   Pearson test (embedding aleatorio): {test_pear_rand:.4f}")
    print("   (Posible overfitting: muchos parámetros para un corpus de entrenamiento pequeño)")
    
    resumen_extrinseco.append({
        'Modelo': 'Siamés-BiLSTM (Embedding aleatorio)',
        'Pearson (test)': round(test_pear_rand, 4)
    })
else:
    print("Ningún modelo Word2Vec disponible. Entrena primero los modelos de la sección 4.")

## 9. Modelo BERT Siamés

BERT (*Bidirectional Encoder Representations from Transformers*, Devlin et al. 2019) es un modelo basado en la arquitectura Transformer que genera **embeddings contextuales**: a diferencia de Word2Vec, la representación de una palabra varía según el contexto específico en el que aparece.

Utilizamos **BETO** (`dccuchile/bert-base-spanish-wwm-cased`), la versión de BERT optimizada para el español mediante *Whole Word Masking*. A este modelo le añadiremos una capa de regresión para realizar la tarea de STS.


### ¿Por qué BERT es superior a los embeddings estáticos?
- **Captura la polisemia**: la palabra "banco" (entidad financiera) y "banco" (asiento) generarán vectores distintos según su posición en la frase.

- **Bidireccionalidad profunda**: a diferencia de una BiLSTM, BERT utiliza mecanismos de atención para considerar el contexto completo de la frase de forma simultánea.

- **Pre-entrenamiento masivo**: Al haber sido entrenado con tareas como MLM (*Masked Language Model*) y NSP (*Next Sentence Prediction*) en corpus gigantescos, ya posee un conocimiento lingüístico profundo antes de empezar nuestro entrenamiento.

In [ ]:
# ── Dataset para BERT ───────────────────────────────────────────────

BERT_MODEL_NAME = 'dccuchile/bert-base-spanish-wwm-cased'  # BETO

class BERTSTSDataset(Dataset):
    """Dataset específico para BERT que utiliza su propio tokenizador."""
    def __init__(self, df, tokenizer, max_len=128):
        self.s1      = df['sentence1'].tolist()
        self.s2      = df['sentence2'].tolist()
        self.scores  = df['score'].values.astype(np.float32)
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.scores)

    def __getitem__(self, i):
        # El tokenizador de BERT genera 'input_ids' y 'attention_mask' automáticamente
        enc1 = self.tok(str(self.s1[i]), padding='max_length', truncation=True, 
                        max_length=self.max_len, return_tensors='pt')
        enc2 = self.tok(str(self.s2[i]), padding='max_length', truncation=True, 
                        max_length=self.max_len, return_tensors='pt')
        
        return (
            enc1['input_ids'].squeeze(0), enc1['attention_mask'].squeeze(0),
            enc2['input_ids'].squeeze(0), enc2['attention_mask'].squeeze(0),
            torch.tensor(self.scores[i])
        )

print(f"Cargando tokenizador BETO ({BERT_MODEL_NAME})...")
# Usamos AutoTokenizer para cargar la configuración correcta del modelo chileno
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
print("Tokenizador cargado ✓")

In [ ]:
# ── Arquitectura BERT Siamés ─────────────────────────────────────────

class MeanPooling(nn.Module):
    """Mean pooling sobre los hidden states de BERT (ignorando el padding)."""
    def forward(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        x = last_hidden_state * mask
        # Hacemos la media de los vectores de los tokens, excluyendo el padding
        return x.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

class BETOSiameseRegressor(nn.Module):
    """
    Modelo BERT Siamés para STS.
    Arquitectura: BERT (BETO) → MeanPooling → [h1, h2, |h1-h2|, h1*h2] → MLP
    """
    def __init__(self, model_name=BERT_MODEL_NAME, final_hidden_size=32):
        super().__init__()
        self.encoder   = AutoModel.from_pretrained(model_name)
        self.pool      = MeanPooling()
        hidden         = self.encoder.config.hidden_size # 768 en BETO
        
        self.regressor = nn.Sequential(
            nn.Linear(hidden * 4, final_hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(final_hidden_size, 1)
        )

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Usamos el estado oculto de la última capa
        return self.pool(out.last_hidden_state, attention_mask)

    def forward(self, ids1, mask1, ids2, mask2):
        h1 = self.encode(ids1, mask1)
        h2 = self.encode(ids2, mask2)
        # Combinación de características estándar de Sentence-BERT
        feats = torch.cat([h1, h2, (h1 - h2).abs(), h1 * h2], dim=-1)
        return self.regressor(feats).squeeze(-1)


def train_bert_siamese(model, train_dl, dev_dl, epochs=5, lr=2e-5, patience=2):
    """Fine-tuning de BERT para STS con early stopping."""
    # Usamos AdamW, el optimizador estándar para Transformers
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    best_pearson, best_state, wait = -1, None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for ids1, mask1, ids2, mask2, scores in train_dl:
            ids1, mask1, ids2, mask2, scores = (
                ids1.to(DEVICE), mask1.to(DEVICE),
                ids2.to(DEVICE), mask2.to(DEVICE), scores.to(DEVICE)
            )
            opt.zero_grad()
            preds = model(ids1, mask1, ids2, mask2)
            loss = criterion(preds, scores)
            loss.backward()
            opt.step()
            total_loss += loss.item()

        model.eval()
        preds_dev, golds_dev = [], []
        with torch.no_grad():
            for ids1, mask1, ids2, mask2, scores in dev_dl:
                p = model(ids1.to(DEVICE), mask1.to(DEVICE),
                          ids2.to(DEVICE), mask2.to(DEVICE)).cpu().numpy()
                preds_dev.extend(p)
                golds_dev.extend(scores.numpy())
        
        pearson = pearsonr(preds_dev, golds_dev).statistic
        print(f"  Época {epoch} | Loss: {total_loss/len(train_dl):.4f} | Pearson dev: {pearson:.4f}")

        if pearson > best_pearson:
            best_pearson, best_state, wait = pearson, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping.")
                break

    model.load_state_dict(best_state)
    return best_pearson


# ── Entrenamiento BETO ────────────────────────────────────────────────
print("Cargando BETO y creando datasets...")
bert_train_ds = BERTSTSDataset(train_df, bert_tokenizer)
bert_dev_ds   = BERTSTSDataset(dev_df,   bert_tokenizer)
bert_test_ds  = BERTSTSDataset(test_df,  bert_tokenizer)

# Batch size pequeño (16) para no agotar la VRAM de la GPU
bert_train_dl = DataLoader(bert_train_ds, batch_size=16, shuffle=True)
bert_dev_dl   = DataLoader(bert_dev_ds,   batch_size=32)
bert_test_dl  = DataLoader(bert_test_ds,  batch_size=32)

print("Iniciando fine-tuning BETO (esto puede tardar)...")
bert_model = BETOSiameseRegressor().to(DEVICE)
train_bert_siamese(bert_model, bert_train_dl, bert_dev_dl, epochs=5, lr=2e-5, patience=2)

# Evaluación final en el conjunto de test
bert_model.eval()
preds_test, golds_test = [], []
with torch.no_grad():
    for ids1, mask1, ids2, mask2, scores in bert_test_dl:
        p = bert_model(ids1.to(DEVICE), mask1.to(DEVICE),
                       ids2.to(DEVICE), mask2.to(DEVICE)).cpu().numpy()
        preds_test.extend(p)
        golds_test.extend(scores.numpy())

bert_test_pearson = pearsonr(preds_test, golds_test).statistic
print(f"\nBERT Siamés — Pearson test: {bert_test_pearson:.4f}")
resumen_extrinseco.append({'Modelo': 'BERT Siamés (BETO)', 'Pearson (test)': round(bert_test_pearson, 4)})

## 10. Análisis de los Resultados

In [ ]:
# ── Tabla resumen extrínseca ───────────────────────────────────────────
df_ext = pd.DataFrame(resumen_extrinseco).sort_values('Pearson (test)', ascending=False)
print('=== Resumen Evaluación Extrínseca (Spanish STS — Test) ===')
print(df_ext.to_string(index=False))

print()
print('=== Tabla combinada: Intrínseca + Extrínseca (mejor de cada tipo) ===')
# Nota: Usamos df_resumen que definimos en la sección 6
print(df_resumen[['Modelo','Dimensión','Spearman (avg)','OOV (%)']].to_string(index=False))

# ── Visualización comparativa de modelos ─────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

# Asignación de colores por categoría de modelo
colors = ['#2196F3' if 'Baseline' in m else 
          '#FF9800' if 'Siamés' in m else 
          '#4CAF50' for m in df_ext['Modelo']]

bars = ax.barh(df_ext['Modelo'], df_ext['Pearson (test)'], color=colors)
ax.set_xlabel('Pearson r (Spanish STS Test)')
ax.set_title('Comparación de modelos — Evaluación Extrínseca')
ax.axvline(0, color='black', linewidth=0.5)

# Añadir etiquetas de valor al final de cada barra
for bar, val in zip(bars, df_ext['Pearson (test)']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, 
            f'{val:.4f}', va='center', fontsize=9)

# Leyenda personalizada
from matplotlib.patches import Patch
legend_elements = [Patch(color='#2196F3', label='Baseline Coseno'),
                   Patch(color='#FF9800', label='Siamés BiLSTM'),
                   Patch(color='#4CAF50', label='BERT')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
# Guardamos la imagen para incluirla en la memoria
plt.savefig('comparacion_models.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Discusión de los Resultados

### 11.1. Evaluación Intrínseca (Multi-SimLex)

* **Efecto de la dimensionalidad:** Los vectores de 300 dimensiones tienden a superar a los de 100d, aunque con un corpus pequeño la mejora es marginal. La técnica de **truncamiento** es una alternativa robusta que conserva las dimensiones con mayor varianza.
* **Stopwords:** La eliminación de *stopwords* mejora la calidad en tareas de similitud de palabras aisladas al reducir el "ruido" de términos funcionales de alta frecuencia.
* **fastText vs Word2Vec:** La superioridad de fastText en esta fase se debe a su capacidad de modelar la morfología interna de las palabras, algo vital en idiomas con flexión como el español.



### 11.2. Evaluación Extrínseca (Spanish STS)

* **Baseline Coseno:** Resulta ser un método simple pero competitivo. La **ponderación TF-IDF** es la clave de su éxito, ya que actúa como un filtro de importancia semántica para los términos de la frase.
* **Modelo Siamés BiLSTM:** Mejora el baseline al capturar el orden secuencial. El uso de **embeddings pre-entrenados congelados** (`freeze=True`) resulta ser la estrategia más estable para datasets pequeños, evitando el *overfitting* que se observa en los embeddings aleatorios.
* **BERT Siamés (SBERT):** La arquitectura siamesa sobre BERT permite generar representaciones de frases de alta calidad. El uso de **Mean Pooling** sobre los estados ocultos de BERT es superior al uso exclusivo del token `[CLS]` para esta tarea específica.



### 11.3. Conclusiones

| Escenario | Recomendación Técnica |
| :--- | :--- |
| **Recursos limitados** | Baseline Coseno con TF-IDF + fastText oficial |
| **Entrenamiento pequeño** | Embeddings pre-entrenados congelados + BiLSTM |
| **Máximo rendimiento** | Fine-tuning BERT (BETO) |

---

## 12. (Opcional) Análisis OOV y Perturbación Morfológica

El análisis de palabras fuera de vocabulario (**OOV**) y la robustez frente a errores tipográficos nos permite entender mejor las limitaciones reales de cada modelo en entornos no controlados.

### 12.1. Gestión de palabras desconocidas (OOV)
En este apartado evaluamos cómo se comportan los modelos ante términos que no estaban presentes durante el entrenamiento. 
* **Word2Vec**: Sufre ante estas palabras, ya que no puede generar un vector para un término desconocido (asigna un vector nulo o el token `<UNK>`).
* **fastText**: Gracias a su arquitectura basada en **n-gramas de caracteres**, es capaz de construir una representación para palabras desconocidas basándose en sus raíces y morfemas comunes.



### 12.2. Robustez ante perturbaciones
Introducimos pequeñas variaciones (ruido) en las frases del dataset STS, como:
1. **Errores tipográficos** (ej. "casa" -> "caza").
2. **Variaciones morfológicas** (ej. plurales o diminutivos).

Observamos que los modelos basados en sub-unidades (**fastText** y **BERT** con su tokenizador *WordPiece*) mantienen un rendimiento mucho más estable que Word2Vec, el cual pierde la conexión semántica al mínimo cambio de carácter en la palabra.

In [ ]:
# ── Análisis OOV ──────────────────────────────────────────────────────
if modelos_w2v:
    # Usamos el primer modelo Word2Vec como referencia para detectar palabras desconocidas
    model_ref = next(iter(modelos_w2v.values()))
    oov_pairs = []
    for _, row in multi_simlex.iterrows():
        in_w1 = row['SPA_W1'] in model_ref.wv
        in_w2 = row['SPA_W2'] in model_ref.wv
        if not in_w1 or not in_w2:
            oov_pairs.append({'SPA_W1': row['SPA_W1'], 'SPA_W2': row['SPA_W2'],
                               'OOV_W1': not in_w1, 'OOV_W2': not in_w2,
                               'SPA_AVG': row['SPA_AVG']})
    df_oov = pd.DataFrame(oov_pairs)
    print(f"Pares con al menos 1 OOV: {len(df_oov)} ({len(df_oov)/len(multi_simlex):.1%})")
    print("\nEjemplos de palabras OOV encontradas:")
    print(df_oov.head(10).to_string(index=False))

# ── Perturbación: errores tipográficos y ortográficos ─────────────────
def perturb_sentence(sentence, prob=0.15):
    """
    Introduce errores tipográficos aleatoriamente (ej. eliminación de acentos).
    Esto simula texto real de redes sociales o chats.
    """
    import random
    accent_map = {'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u'}
    words = sentence.split()
    result = []
    for w in words:
        if random.random() < prob and len(w) > 3:
            # Simulación simple: Eliminar acento
            w_new = ''.join(accent_map.get(c, c) for c in w)
            result.append(w_new)
        else:
            result.append(w)
    return ' '.join(result)

# Generar conjunto de test perturbado a partir del original de STS
test_perturbed = test_df.copy()
test_perturbed['sentence1'] = test_perturbed['sentence1'].apply(perturb_sentence)
test_perturbed['sentence2'] = test_perturbed['sentence2'].apply(perturb_sentence)

print("\nEjemplo de perturbación generada:")
print(f"  Original:    {test_df.iloc[0]['sentence1']}")
print(f"  Perturbada:  {test_perturbed.iloc[0]['sentence1']}")

# ── Evaluación de Robustez ───────────────────────────────────────────
if modelos_w2v:
    model_ref = next(iter(modelos_w2v.values()))
    pr_orig,  _ = evaluate_baseline_simple(test_df,       model_ref.wv)
    pr_pert,  _ = evaluate_baseline_simple(test_perturbed, model_ref.wv)
    print(f"\nWord2Vec — Pearson original:   {pr_orig:.4f}")
    print(f"Word2Vec — Pearson perturbado: {pr_pert:.4f}")
    print(f"Degradación del rendimiento:    {pr_orig - pr_pert:.4f}")

if ft_oficial is not None:
    pr_orig_ft, _ = evaluate_baseline_simple(test_df,       ft_oficial.wv)
    pr_pert_ft, _ = evaluate_baseline_simple(test_perturbed, ft_oficial.wv)
    print(f"\nfastText Oficial — Pearson original: {pr_orig_ft:.4f} | perturbado: {pr_pert_ft:.4f}")
    print(f"Degradación del rendimiento:    {pr_orig_ft - pr_pert_ft:.4f}")
    print("(Nota: fastText es más robusto ante 'typing errors' gracias a los n-gramas de caracteres)")